# Fase 4 — Ferramentas Externas e RAG Existente: Ampliando o Alcance do Agente

Esta fase dá rigor às ferramentas do agente e à recuperação de conhecimento:

1. ferramentas documentadas por tipo (*hardcoded*, semântica, API, *text-to-query*, RAG, ação);
2. integração com APIs conforme o tipo de uso de cada uma;
3. padrão *text-to-query* sobre o backlog, com *guardrails* e autocorreção;
4. agente de recuperação sobre o **índice FAISS já existente** do Manual de Engenharia;
5. **diagnóstico de falhas reais de recuperação** a partir dos documentos-fonte devolvidos, e **correção por transformação de consulta**, medida contra um gabarito.

In [1]:
import sys, pathlib, json, os, warnings
import pandas as pd
RAIZ = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ))
from fabrica_agentica.config import descrever_ambiente, texto_de
print(descrever_ambiente())
pd.set_option("display.max_colwidth", 90)

Provedor de LLM : anthropic
APIs externas   : rede (ao vivo)
Embeddings      : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Modelo NLI      : MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli
LangSmith       : ligado
Repositório GH  : DriccoC/fabrica-agentica-sandbox


## 1. Ferramentas documentadas por tipo

Cada ferramenta tem os três pilares: **nome**, **descrição/esquema** (a interface que o LLM lê para decidir quando e como chamar — por isso as descrições são escritas para o modelo, com regras como "nunca calcule de cabeça") e **lógica central**. A tabela é gerada a partir do próprio código, para não divergir dele.

In [2]:
from fabrica_agentica.ferramentas.catalogo import FERRAMENTAS, TIPOS

tabela = pd.DataFrame([
    {"tipo": TIPOS[t.name], "ferramenta": t.name, "argumentos": ", ".join(t.args.keys()),
     "descrição (início)": " ".join(t.description.split())[:110]}
    for t in FERRAMENTAS
]).sort_values("tipo")
tabela

,tipo,ferramenta,argumentos,descrição (início)
3,API externa,consultar_feriados,ano,Lista os feriados nacionais brasileiros de um ano (API pública BrasilAPI).
4,API externa,consultar_pypi,pacote,Consulta no PyPI (API pública) a versão estável mais recente de um pacote Python e as ...
5,API externa,consultar_vulnerabilidades,"pacote, versao",Consulta na base pública OSV.dev as vulnerabilidades conhecidas que afetam uma versão ...
6,API externa,buscar_issues_github,termo,Busca issues (abertas ou fechadas) no repositório GitHub do time que mencionem o termo...
8,RAG (FAISS),buscar_manual_engenharia,"consulta_otimizada, documento","Busca no Manual de Engenharia (normas MAN-01 a MAN-07: Definição de Pronto, padrões de..."
11,ação (aprovação humana),propor_registro_backlog,"projeto, numero_sprint, plano, abrir_issues",PROPÕE o registro das histórias no backlog (e a abertura de issues no GitHub). Não gra...
0,hardcoded,calcular_capacidade_sprint,"projeto, numero_sprint, remover_integrantes",Calcula a capacidade de pontos de uma sprint pela fórmula do MAN-06 (dias úteis sem fe...
1,hardcoded,comparar_versoes,"versao_atual, versao_mais_recente",Compara uma versão em uso com a mais recente (versionamento semântico) e diz se está c...
2,hardcoded,classificar_cvss,pontuacao,Classifica uma pontuação CVSS v3 (0 a 10) na faixa de severidade do MAN-04 e diz se a ...
9,semântica,decompor_demanda,"demanda, contexto","Decompõe uma demanda em épico, histórias de usuário (com critérios de aceite em Gherki..."


| Tipo | Lógica central | Quando usar | No projeto |
|---|---|---|---|
| **Hardcoded** (determinística) | código fixo; mesma entrada, mesma saída, sem tokens | a precisão é inegociável e a regra é conhecida | capacidade da sprint (fórmula do MAN-06), comparação semver, faixa CVSS (MAN-04) |
| **Semântica** | uma sub-chamada ao LLM com prompt especialista e saída estruturada | exige interpretação ou geração de linguagem | decompor a demanda em épico/histórias/Gherkin, resumir textos técnicos |
| **API externa** | chamada HTTP a um sistema fora do LLM | o dado muda com o tempo ou vive em outro sistema | PyPI, OSV.dev, BrasilAPI, GitHub |
| **Text-to-query** | o LLM gera SQL; o código valida e executa | dados estruturados e consultas variáveis | backlog (projetos, sprints, histórias, bugs) |
| **RAG** | busca vetorial no índice FAISS | conhecimento normativo em texto | Manual de Engenharia |
| **Ação** | escrita com efeito externo, só após aprovação humana | registrar trabalho, abrir issues | `propor_registro_backlog` + `acoes.executar_registro` |

### Determinística × semântica, na prática

In [3]:
from fabrica_agentica.ferramentas.catalogo import calcular_capacidade_sprint, resumir_texto_tecnico

a = calcular_capacidade_sprint.invoke({"projeto": "Portal"})
b = calcular_capacidade_sprint.invoke({"projeto": "Portal"})
print("Hardcoded — duas chamadas idênticas?", a == b, "| saldo:", json.loads(a)["saldo_disponivel"])

texto = ("A sprint 13 do Portal tem 9 dias úteis por causa do feriado de 20/11. A capacidade líquida é 25 e já há 11 "
         "pontos comprometidos. O login com Google exige OIDC com PKCE e revisão de segurança.")
r1 = resumir_texto_tecnico.invoke({"texto": texto, "foco": "riscos"})
r2 = resumir_texto_tecnico.invoke({"texto": texto, "foco": "riscos"})
print("Semântica — duas chamadas idênticas?", r1 == r2)
print("--- resumo 1 ---\n", r1, "\n--- resumo 2 ---\n", r2)

Hardcoded — duas chamadas idênticas? True | saldo: 14


Semântica — duas chamadas idênticas? False
--- resumo 1 ---
 - **Prazo reduzido:** a sprint tem 9 dias úteis por causa do feriado de 20/11, o que pode pressionar a entrega.
- **Capacidade parcial:** com 11 pontos já comprometidos de uma capacidade líquida de 25, restam 14 pontos para novas entregas.
- **Dependência de revisão de segurança:** o login com Google exige revisão de segurança, que pode atrasar a conclusão dessa funcionalidade.
- **Complexidade técnica:** a implementação exige OIDC com PKCE, o que pode aumentar o esforço e o risco de erros de configuração. 
--- resumo 2 ---
 - A sprint 13 terá apenas 9 dias úteis devido ao feriado de 20/11, o que reduz o tempo disponível para entrega.
- A capacidade líquida é de 25 pontos, mas já há 11 pontos comprometidos antes do início do planejamento adicional.
- A implementação do login com Google exige OIDC com PKCE, o que aumenta a complexidade técnica.
- O login com Google depende de revisão de segurança, que pode gerar atrasos caso s

## 2. Integração com APIs conforme o tipo de uso

| API | Tipo de uso | Decisões de integração |
|---|---|---|
| **PyPI JSON** (pypi.org) | web API pública, leitura | sem chave; *timeout* de 10 s; filtra versões pré-lançamento; pacote inexistente devolve `encontrado: false` em vez de exceção |
| **OSV.dev** | web API pública, leitura (POST de consulta) | sem chave; calcula a pontuação CVSS a partir do vetor (`cvss`); devolve só as 5 mais graves ao LLM para não estourar o contexto |
| **BrasilAPI** (feriados) | web API pública, leitura, insumo de cálculo | usada *dentro* de uma ferramenta hardcoded (capacidade); se falhar, o cálculo segue sem feriados e **informa** a fonte indisponível |
| **GitHub REST** | web API autenticada; leitura e **escrita** | leitura por busca; escrita (abrir issue) só via ação aprovada por humano, com token *fine-grained* restrito a um repositório; pausa de 1 s entre criações (limite de conteúdo) |
| **Backlog (SQLite)** | API interna de dados (*text-to-query*) | conexão somente leitura para o agente; a escrita acontece fora do agente, após aprovação |

Funções *serverless* não são usadas: as integrações são síncronas e leves, e o agente roda num único processo. Em todas as APIs, erros viram **JSON de erro devolvido ao LLM** (que pode explicar ou tentar outro caminho) em vez de derrubar a execução. Para avaliação sem rede ou sem credenciais, `APIS_OFFLINE=1` troca as chamadas por respostas gravadas.

In [4]:
from fabrica_agentica.ferramentas.catalogo import consultar_pypi, consultar_vulnerabilidades, consultar_feriados, buscar_issues_github

print("PyPI (ao vivo):", consultar_pypi.invoke({"pacote": "fastapi"})[:160])
print("Pacote inexistente:", consultar_pypi.invoke({"pacote": "pacote-que-nao-existe-xyz-123"}))
osv = json.loads(consultar_vulnerabilidades.invoke({"pacote": "python-jose", "versao": "3.3.0"}))
print(f"OSV: python-jose 3.3.0 -> {osv['total']} vulnerabilidades, maior CVSS {osv['maior_cvss']}")
print("BrasilAPI:", [f for f in json.loads(consultar_feriados.invoke({"ano": 2026})) if f["date"].startswith("2026-11")])
print("GitHub (leitura):", buscar_issues_github.invoke({"termo": "login"})[:200])

os.environ["APIS_OFFLINE"] = "1"
print("\nModo offline (respostas gravadas):", consultar_pypi.invoke({"pacote": "authlib"})[:120])
del os.environ["APIS_OFFLINE"]

PyPI (ao vivo): {"pacote": "fastapi", "encontrado": true, "versao_mais_recente": "0.162.0", "resumo": "FastAPI framework, high performance, easy to learn, fast to code, ready f


Pacote inexistente: {"pacote": "pacote-que-nao-existe-xyz-123", "encontrado": false}


OSV: python-jose 3.3.0 -> 7 vulnerabilidades, maior CVSS 9.3


BrasilAPI: [{'date': '2026-11-02', 'name': 'Finados'}, {'date': '2026-11-15', 'name': 'Proclamação da República'}, {'date': '2026-11-20', 'name': 'Dia da consciência negra'}]


GitHub (leitura): {"repositorio": "DriccoC/fabrica-agentica-sandbox", "modo": "ao vivo", "issues": []}

Modo offline (respostas gravadas): {"pacote": "authlib", "encontrado": true, "versao_mais_recente": "1.9.5", "resumo": "The ultimate Python library in buil


## 3. Padrão *text-to-query*

A pergunta em linguagem natural vira uma consulta SQL executável sobre o backlog. O LLM recebe o **dicionário de dados** (tabelas, colunas, valores válidos e relacionamentos) e devolve a consulta com saída estruturada; o código valida e executa. Se der erro de SQL, o erro volta ao LLM para uma nova versão (autocorreção).

In [5]:
from fabrica_agentica.ferramentas.sql import DICIONARIO_DADOS
print(DICIONARIO_DADOS)

Banco de backlog da NovaForja Software (SQLite). Tabelas e colunas:
- projetos(projeto_id, nome, descricao, stack, repositorio)  -- nomes: 'Portal', 'PagaFacil', 'AppCampo'
- desenvolvedores(dev_id, nome, papel, senioridade)
- sprints(sprint_id, projeto_id, numero, inicio, fim, status)  -- status: concluida | em_andamento | planejada; datas AAAA-MM-DD
- alocacoes(sprint_id, dev_id, fator_foco, dias_ausencia)  -- quem está em cada sprint
- historias(historia_id, projeto_id, sprint_id, epico, titulo, pontos, prioridade, status, responsavel_id, criado_em, updated_at)
    -- sprint_id NULL = backlog sem sprint; status: backlog | a_fazer | em_andamento | concluida; prioridade: alta | media | baixa
- bugs(bug_id, projeto_id, titulo, severidade, status, aberto_em)  -- severidade: critica | alta | media | baixa
Relacionamentos: sprints.projeto_id -> projetos; historias.sprint_id -> sprints; alocacoes liga sprints e desenvolvedores.
Observação: 'sprints.numero' é o número exibido ao time (ex.: 

In [6]:
from fabrica_agentica.chains.text_to_query import pergunta_para_sql

perguntas = [
    "Qual foi a velocidade (pontos concluídos) de cada sprint do Portal?",
    "Quais bugs críticos ou de severidade alta ainda não foram resolvidos, e em que projeto?",
    "Quem está alocado na sprint 13 do Portal, com qual fator de foco e quantos dias de ausência?",
    "Quais histórias de autenticação estão no backlog sem sprint, e quantos pontos somam?",
    "Quais histórias foram atualizadas depois de 01/11/2026?",
]
for p in perguntas:
    r = pergunta_para_sql(p)
    print(f"\n❓ {p}\n   raciocínio: {r['raciocinio']}")
    for t in r["tentativas"]:
        print(f"   tentativa {t['tentativa']}: {'OK' if t['ok'] else 'ERRO'} | {t['sql']}")
    print(f"   💬 {r['resposta']}")


❓ Qual foi a velocidade (pontos concluídos) de cada sprint do Portal?
   raciocínio: Somar pontos das historias concluidas agrupadas por sprint, juntando sprints e projetos para filtrar o projeto 'Portal'.
   tentativa 1: OK | SELECT s.numero AS sprint_numero, s.inicio, s.fim, s.status AS status_sprint, COALESCE(SUM(h.pontos), 0) AS pontos_concluidos FROM sprints s JOIN projetos p ON p.projeto_id = s.projeto_id LEFT JOIN historias h ON h.sprint_id = s.sprint_id AND h.status = 'concluida' WHERE p.nome = 'Portal' GROUP BY s.sprint_id, s.numero, s.inicio, s.fim, s.status ORDER BY s.numero;
   💬 A Sprint 11 concluiu 21 pontos, a Sprint 12 (em andamento) tem 5 pontos concluídos até agora, e a Sprint 13 (planejada) ainda não tem pontos concluídos (0).



❓ Quais bugs críticos ou de severidade alta ainda não foram resolvidos, e em que projeto?
   raciocínio: Filtrar a tabela bugs por severidade critica ou alta e status ainda não resolvido, juntando com projetos pelo projeto_id para obter o nome do projeto.
   tentativa 1: OK | SELECT b.bug_id, b.titulo, b.severidade, b.status, b.aberto_em, p.nome AS projeto FROM bugs b JOIN projetos p ON p.projeto_id = b.projeto_id WHERE b.severidade IN ('critica', 'alta') AND b.status NOT IN ('resolvido', 'fechado', 'concluido') ORDER BY CASE b.severidade WHEN 'critica' THEN 1 ELSE 2 END, b.aberto_em;
   💬 Há dois bugs críticos ou de severidade alta ainda não resolvidos. O bug #4, "Duplicidade em conciliação de boletos de fim de mês" (severidade crítica, status em correção), é do projeto PagaFacil. O bug #1, "Sessão expira antes do tempo configurado" (severidade alta, status aberto), é do projeto Portal.



❓ Quem está alocado na sprint 13 do Portal, com qual fator de foco e quantos dias de ausência?
   raciocínio: Juntar alocacoes a desenvolvedores e sprints, filtrando pelo numero 13 da sprint e pelo projeto Portal via projetos.
   tentativa 1: OK | SELECT d.nome, a.fator_foco, a.dias_ausencia FROM alocacoes a JOIN desenvolvedores d ON d.dev_id = a.dev_id JOIN sprints s ON s.sprint_id = a.sprint_id JOIN projetos p ON p.projeto_id = s.projeto_id WHERE s.numero = 13 AND p.nome = 'Portal';
   💬 Estão alocados na sprint 13 do Portal: Ana Souza (fator de foco 0,7, 0 dias de ausência), Bruno Lima (fator 0,8, 2 dias de ausência), Carla Mendes (fator 0,8, 0 dias), Diego Rocha (fator 0,6, 0 dias) e Eduarda Nunes (fator 0,75, 0 dias).



❓ Quais histórias de autenticação estão no backlog sem sprint, e quantos pontos somam?
   raciocínio: Filtrar historias com sprint_id NULL e status backlog, buscando titulos ou epicos relacionados a autenticacao, e somar os pontos.
   tentativa 1: OK | SELECT historia_id, epico, titulo, pontos, SUM(pontos) OVER () AS total_pontos FROM historias WHERE sprint_id IS NULL AND status = 'backlog' AND (LOWER(titulo) LIKE '%autentica%' OR LOWER(epico) LIKE '%autentica%');
   💬 Há duas histórias de autenticação no backlog sem sprint: a #131, "Integração com SSO corporativo dos clientes B2B" (13 pontos), e a #132, "Autenticação em dois fatores (MFA) por aplicativo" (8 pontos). Juntas, somam 21 pontos.



❓ Quais histórias foram atualizadas depois de 01/11/2026?
   raciocínio: Consulta direta na tabela historias filtrando updated_at posterior a 2026-11-01, no formato ISO usado pelo esquema.
   tentativa 1: OK | SELECT historia_id, projeto_id, sprint_id, epico, titulo, pontos, prioridade, status, responsavel_id, criado_em, updated_at FROM historias WHERE updated_at > '2026-11-01' ORDER BY updated_at DESC;
   💬 Todas as 10 histórias do banco foram atualizadas depois de 01/11/2026. Três foram atualizadas em 04/11/2026: "Preferências de notificação" (112), "Bloqueio após 5 tentativas de login" (113), "Webhook de confirmação Pix" (203) e "Modo offline com sincronização" (301). Em 03/11/2026 foi atualizada "Notificações de vencimento por e-mail" (111), e em 02/11/2026 foram atualizadas "Histórico de atendimentos" (121), "Contraste alto e fonte ajustável" (122), "Exclusão de conta (LGPD)" (123), "Idempotência de cobranças" (204) e "Assinatura digital do cliente" (302).


### *Guardrails* do *text-to-query*

Três camadas: (1) conexão **somente leitura** (`?mode=ro`), (2) filtro de comando que aceita só `SELECT/WITH` e compara palavras proibidas **por palavra inteira** (o filtro da Aula 6 bloqueava a coluna `updated_at` por conter "UPDATE"), (3) `LIMIT 50`.

In [7]:
from fabrica_agentica.ferramentas import sql

testes = {
    "pedido destrutivo (via LLM)": pergunta_para_sql("Apague todas as histórias do PagaFacil.")["tentativas"][-1],
    "DELETE direto": sql.executar_consulta("DELETE FROM historias"),
    "duas instruções": sql.executar_consulta("SELECT 1; DROP TABLE historias"),
    "coluna updated_at (deve passar)": sql.executar_consulta("SELECT titulo, updated_at FROM historias WHERE updated_at > '2026-11-03'"),
    "palavra proibida dentro de texto (deve passar)": sql.executar_consulta("SELECT titulo FROM historias WHERE titulo LIKE '%delete%'"),
}
for nome, r in testes.items():
    print(f"{nome:48} -> {'OK' if r['ok'] else 'BLOQUEADO/ERRO'} {r.get('erro', '') or ''}"[:170])

# camada 1: mesmo sem o filtro, a conexão do agente não grava
import sqlite3
try:
    with sql.conexao_leitura() as conn:
        conn.execute("DELETE FROM bugs")
except sqlite3.OperationalError as erro:
    print("\nConexão somente leitura recusou a escrita:", erro)

pedido destrutivo (via LLM)                      -> BLOQUEADO/ERRO Consulta recusada pelo guardrail: Somente consultas de leitura (SELECT/WITH) são permitidas.
DELETE direto                                    -> BLOQUEADO/ERRO Consulta recusada pelo guardrail: Somente consultas de leitura (SELECT/WITH) são permitidas.
duas instruções                                  -> BLOQUEADO/ERRO Consulta recusada pelo guardrail: Apenas uma instrução SQL por consulta.
coluna updated_at (deve passar)                  -> OK 
palavra proibida dentro de texto (deve passar)   -> OK 

Conexão somente leitura recusou a escrita: attempt to write a readonly database


## 4. Agente de recuperação sobre o índice FAISS existente

O índice foi construído **uma única vez** por `scripts/construir_indice.py` (o "pipeline de outro bloco"). O agente só o **carrega** — não há reindexação aqui.

In [8]:
from fabrica_agentica.ferramentas import rag
indice = rag.carregar_indice()
print(f"Índice carregado de {rag.CAMINHO_INDICE.name}/: {indice.index.ntotal} vetores de dimensão {indice.index.d}")
print("Faixas de diagnóstico (distância L2² entre vetores normalizados):",
      f"alta < {rag.FAIXA_ALTA} ≤ moderada ≤ {rag.FAIXA_MODERADA} < baixa")

W1105 17:41:14.271000 81864 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Índice carregado de indice_faiss/: 42 vetores de dimensão 384
Faixas de diagnóstico (distância L2² entre vetores normalizados): alta < 0.8 ≤ moderada ≤ 1.1 < baixa


### 4.1 Casos difíceis com gabarito

14 perguntas reais do dia a dia do time, em jargão, cada uma com o *chunk* que contém a resposta (gabarito). Métricas: **hit@1** (o trecho certo veio em 1º), **hit@3** (entre os 3 primeiros — o que o agente efetivamente lê) e **MRR@4**.

In [9]:
from fabrica_agentica.ferramentas import transformacao_consulta as tc

base = tc.avaliar(tc.busca_direta)
print(f"SEM transformação: hit@1={base['hit@1']} | hit@3={base['hit@3']} | MRR@4={base['mrr@4']}")
pd.DataFrame(base["linhas"])

SEM transformação: hit@1=0.357 | hit@3=0.714 | MRR@4=0.542


,pergunta,posicao,top1,dist_top1
0,quem precisa revisar meu PR?,4.0,CHK_030 MAN-06 MAN-06 — Norma de Planejamen,1.224
1,posso commitar direto no master?,NaN,CHK_034 MAN-06 4. Priorização,1.472
2,posso subir direto na main na sexta?,2.0,CHK_031 MAN-06 1. Iterações,1.308
3,qual a cobertura mínima de testes?,2.0,CHK_020 MAN-04 2. Critérios de admissão de,0.944
4,posso usar uma lib com CVE alta?,1.0,CHK_019 MAN-04 2. Critérios de admissão de,1.467
5,dá pra fazer deploy sexta à tarde?,2.0,CHK_031 MAN-06 1. Iterações,1.022
6,"o cliente errou a senha várias vezes, o que acontece?",1.0,CHK_027 MAN-05 4. Tokens e sessões,0.926
7,uma história de 13 pontos pode entrar na sprint?,3.0,CHK_033 MAN-06 3. Cálculo de capacidade da,0.694
8,o login com Google tem que passar pelo Keycloak?,1.0,CHK_039 MAN-07 ADR-012 — Provedor de identi,0.887
9,quanto tempo pode durar um branch?,2.0,CHK_031 MAN-06 1. Iterações,1.123


## 5. Diagnóstico de falhas reais a partir dos documentos-fonte

Pegamos duas perguntas em que a busca direta falhou e olhamos **o que o agente de recuperação efetivamente recebeu**.

In [10]:
from fabrica_agentica.agentes.recuperacao import criar_agente_recuperacao, fontes

agente_literal = criar_agente_recuperacao(com_transformacao=False)
casos_falha = ["posso commitar direto no master?", "precisa de dois fatores pra pagar fatura?"]
for pergunta in casos_falha:
    r = agente_literal.invoke({"input": pergunta})
    print(f"\n❓ {pergunta}\nDocumentos-fonte devolvidos ao agente:")
    for f in fontes(r):
        print(f"   {f['distancia']:.3f} [{f['similaridade']:8}] {f['chunk_id']} {f['doc']} | {f['secao'][:35]} | {f['inicio'][:70]}")
    print("🤖", texto_de(r["output"])[:500])


❓ posso commitar direto no master?
Documentos-fonte devolvidos ao agente:
   1.472 [baixa   ] CHK_034 MAN-06 | 4. Priorização | ## 4. Priorização - Defeitos de severidade crítica têm precedência sob
   1.477 [baixa   ] CHK_013 MAN-03 | 2. Ramificações de curta duração | ## 2. Ramificações de curta duração Ramificações de trabalho devem ter
   1.499 [baixa   ] CHK_023 MAN-05 | MAN-05 — Padrão de Autenticação e G | # MAN-05 — Padrão de Autenticação e Gestão de Identidade  Código: MAN-
🤖 O manual não cobre essa pergunta. Os trechos devolvidos não dizem se é permitido commitar direto no master.

Os trechos mais próximos tratam de outros pontos:
- **MAN-03, seção 2 (CHK_013):** as ramificações de trabalho devem durar no máximo dois dias úteis. Funcionalidades incompletas devem ficar ocultas por feature flags até a liberação. O trecho não fala de commits diretos no master.
- **MAN-06, seção 4 (CHK_034):** trata de priorização do backlog. Não tem relação com a pergunta.
- **MAN-05 (



❓ precisa de dois fatores pra pagar fatura?
Documentos-fonte devolvidos ao agente:
   1.144 [baixa   ] CHK_038 MAN-07 | ADR-009 — Mensageria com Kafka para | ## ADR-009 — Mensageria com Kafka para eventos de pagamento Status: ac
   1.326 [baixa   ] CHK_034 MAN-06 | 4. Priorização | ## 4. Priorização - Defeitos de severidade crítica têm precedência sob
   1.464 [baixa   ] CHK_003 MAN-01 | 2. Critérios obrigatórios de conclu | 4. Revisão de código aprovada por pelo menos dois revisores, sendo ao 
🤖 O manual não cobre esse assunto. Os trechos retornados não dizem se é preciso autenticação de dois fatores (2FA) para pagar uma fatura.

Os trechos encontrados tratam de outros temas:
- **MAN-07, ADR-009 (CHK_038):** eventos de pagamento são publicados em tópicos Kafka, e os consumidores devem ser idempotentes.
- **MAN-06, seção 4 (CHK_034):** regras de priorização do backlog.
- **MAN-01, seção 2 (CHK_003):** critérios de conclusão, como revisão de código.

Nenhum deles fala de autenticação o

In [11]:
# Onde estava a resposta certa? Distância da pergunta literal até o chunk do gabarito
catalogo = {d.metadata["chunk_id"]: d for d in indice.docstore._dict.values()}
for pergunta, gabarito in [("posso commitar direto no master?", "CHK_012"), ("precisa de dois fatores pra pagar fatura?", "CHK_028")]:
    todos = rag.buscar(pergunta, k=42)
    pos = next(i for i, r in enumerate(todos, 1) if r["chunk_id"] == gabarito)
    print(f"'{pergunta}': o gabarito {gabarito} ficou em {pos}º de 42 (distância {todos[pos-1]['distancia_l2']})")
    print(f"   texto do gabarito: {catalogo[gabarito].page_content[:260]!r}\n")

'posso commitar direto no master?': o gabarito CHK_012 ficou em 7º de 42 (distância 1.564)
   texto do gabarito: '## 1. Modelo de ramificação\nAdota-se o desenvolvimento baseado em tronco (trunk-based development). O ramo principal é protegido: nenhuma alteração pode ser incorporada a ele por envio direto. Toda integração ao tronco ocorre exclusivamente por meio de solicit'

'precisa de dois fatores pra pagar fatura?': o gabarito CHK_028 ficou em 12º de 42 (distância 1.651)
   texto do gabarito: '## 5. Autenticação multifator\nA autenticação multifator é obrigatória para operações sensíveis (alteração de dados cadastrais, pagamentos) e para todos os usuários administrativos.'



**Diagnóstico:**

1. **Descompasso de vocabulário.** O usuário fala em "commitar direto no master"; a norma diz "o ramo principal é protegido: nenhuma alteração pode ser incorporada a ele por envio direto" (MAN-03 §1). Nenhuma palavra relevante em comum — "commitar" e "master" nem aparecem no manual. Idem "dois fatores" × "autenticação multifator" (MAN-05 §5).
2. **Atração por palavras secundárias.** Sem os termos certos, a busca se apega ao que sobra: "posso", "direto", "pagar fatura" — e traz Priorização (MAN-06 §4) ou trechos de pagamentos/sessões, todos com similaridade **baixa** (distância > 1,1).
3. **Trecho do gabarito com dois assuntos.** O `CHK_012` começa por "desenvolvimento baseado em tronco (trunk-based development)", e a frase sobre o ramo protegido fica diluída no vetor do *chunk*. Corrigir isso reindexando (*chunks* menores) está fora do escopo — o índice é um ativo existente —, então a correção tem de ser **do lado da consulta**.
4. **O agente não alucinou:** com trechos de baixa similaridade, ele declarou que o manual não cobre o assunto, como manda o prompt. A falha é de **recuperação**, não de geração — e é por isso que o diagnóstico olha os documentos-fonte, e não só a resposta.

## 6. Correção: transformação de consulta, medida contra o gabarito

Comparamos as três técnicas pedidas no enunciado (reescrita, múltiplas consultas e documento hipotético/HyDE) e a combinação multi-query + HyDE, em **duas rodadas** (as transformações usam LLM e variam entre execuções).

In [12]:
linhas = []
for nome, estrategia in tc.ESTRATEGIAS.items():
    for rodada in (1, 2):
        if nome.startswith("direta") and rodada == 2:
            continue  # a busca direta é determinística
        r = tc.avaliar(estrategia)
        falhas = [l["pergunta"] for l in r["linhas"] if not l["posicao"] or l["posicao"] > 3]
        linhas.append({"estratégia": nome, "rodada": rodada, "hit@1": r["hit@1"], "hit@3": r["hit@3"], "MRR@4": r["mrr@4"],
                       "falhas no top-3": "; ".join(falhas) or "—"})
comparacao = pd.DataFrame(linhas)
comparacao

,estratégia,rodada,hit@1,hit@3,MRR@4,falhas no top-3
0,direta (sem transformação),1,0.357,0.714,0.542,quem precisa revisar meu PR?; posso commitar direto no master?; precisa de dois fatore...
1,reescrita,1,0.571,0.714,0.661,"posso commitar direto no master?; o cliente errou a senha várias vezes, o que acontece..."
2,reescrita,2,0.500,0.643,0.595,qual a cobertura mínima de testes?; posso usar uma lib com CVE alta?; o cliente errou ...
3,multi-query,1,0.571,0.786,0.685,posso commitar direto no master?; uma história de 13 pontos pode entrar na sprint?; qu...
4,multi-query,2,0.643,0.786,0.720,posso commitar direto no master?; quanto tempo pode durar um branch?; toda lib nova te...
5,HyDE,1,0.643,0.929,0.774,toda lib nova tem que ir pro SBOM?
6,HyDE,2,0.714,0.929,0.821,toda lib nova tem que ir pro SBOM?
7,multi-query + HyDE,1,0.786,0.929,0.857,toda lib nova tem que ir pro SBOM?
8,multi-query + HyDE,2,0.714,0.929,0.810,posso subir direto na main na sexta?


In [13]:
comparacao.groupby("estratégia")[["hit@1", "hit@3", "MRR@4"]].agg(["mean", "min"]).round(3)

hit@1         hit@3         MRR@4       
                             mean    min   mean    min   mean    min
estratégia                                                          
HyDE                        0.678  0.643  0.929  0.929  0.798  0.774
direta (sem transformação)  0.357  0.357  0.714  0.714  0.542  0.542
multi-query                 0.607  0.571  0.786  0.786  0.702  0.685
multi-query + HyDE          0.750  0.714  0.929  0.929  0.834  0.810
reescrita                   0.536  0.500  0.678  0.643  0.628  0.595

### 6.1 O agente de recuperação com a correção aplicada

In [14]:
agente_corrigido = criar_agente_recuperacao(com_transformacao=True)
for pergunta in casos_falha:
    r = agente_corrigido.invoke({"input": pergunta})
    passos = r["intermediate_steps"]
    consultas = json.loads(passos[0][1])["consultas_usadas"] if passos else []
    print(f"\n❓ {pergunta}\nConsultas geradas pela transformação:")
    for c in consultas:
        print(f"   - {c[:150]}")
    print("Documentos-fonte devolvidos ao agente:")
    for f in fontes(r):
        print(f"   {f['distancia']:.3f} [{f['similaridade']:8}] {f['chunk_id']} {f['doc']} | {f['secao'][:35]}")
    print("🤖", texto_de(r["output"])[:600])


❓ posso commitar direto no master?
Consultas geradas pela transformação:
   - posso commitar direto no master?
   - É permitido efetuar confirmação de alterações (commit) diretamente no ramo principal (master) do repositório de código-fonte?
   - Quais são as diretrizes normativas para a submissão de alterações (commit) diretamente ao ramo principal (master) em um sistema de controle de versão?
   - Há restrição quanto à integração de modificações (commit) diretamente no ramo mestre (master) do repositório de software?
   - **Norma de Controle de Versão:** Não é permitido realizar commits diretamente na branch `master`, que deve ser protegida contra escrita direta e alter
Documentos-fonte devolvidos ao agente:
   0.773 [alta    ] CHK_012 MAN-03 | 1. Modelo de ramificação
   0.856 [moderada] CHK_004 MAN-01 | 3. Critérios específicos por tipo d
   0.876 [moderada] CHK_003 MAN-01 | 2. Critérios obrigatórios de conclu
🤖 **Não, você não pode commitar direto no master.** O manual diz isso n


❓ precisa de dois fatores pra pagar fatura?
Consultas geradas pela transformação:
   - precisa de dois fatores pra pagar fatura?
   - Requisitos de autenticação de dois fatores (2FA) para autorização de pagamento de faturas em sistemas de software
   - Implementação de autenticação multifator (MFA) em fluxos de pagamento de faturas
   - Exigência de segundo fator de verificação na conclusão de transações de quitação de faturas
   - A autenticação de dois fatores (2FA) é obrigatória para a autorização de pagamentos de faturas em todos os sistemas corporativos, em conformidade com 
Documentos-fonte devolvidos ao agente:
   0.590 [alta    ] CHK_028 MAN-05 | 5. Autenticação multifator
   0.821 [moderada] CHK_004 MAN-01 | 3. Critérios específicos por tipo d
   0.844 [moderada] CHK_025 MAN-05 | 2. Provedores de identidade federad
🤖 Sim, de acordo com o manual, pagar fatura exige autenticação multifator.

O manual diz que "a autenticação multifator é obrigatória para operações sensíveis (alt

### 6.2 Justificativa da técnica adotada

- **Reescrita** resolve o descompasso de vocabulário na maior parte dos casos, mas depende de uma única formulação: quando ela erra o termo, a busca erra junto.
- **Multi-query** dá robustez (várias formulações + a pergunta original) e eleva o *hit@3*, mas o *hit@1* oscila entre rodadas.
- **HyDE** aproxima a consulta do **estilo** do texto indexado (um parágrafo normativo), o que ajuda justamente nos *chunks* com mais de um assunto, como o `CHK_012`; sozinho, porém, falha quando o documento hipotético "inventa" um enquadramento diferente.
- **Combinação multi-query + HyDE** junta os dois ganhos: as variações cobrem o vocabulário, o documento hipotético cobre o estilo, e a união pelo menor *score* mantém o melhor trecho de cada uma. Na execução deste notebook teve o **melhor hit@1 e o melhor MRR médios** (0,750 e 0,834, contra 0,357 e 0,542 sem transformação) e **empatou com HyDE no hit@3** (0,929; a falha restante mudou entre as rodadas: "toda lib nova tem que ir pro SBOM?" numa, "posso subir direto na main na sexta?" na outra); nas rodadas feitas durante o desenvolvimento chegou a hit@3 = 1,0 nas duas. A escolha considera a média e o pior caso de cada técnica, não uma rodada isolada.

**Adotada no agente de recuperação** (que recebe a pergunta literal do usuário). *Atualização após a Fase 5:* no agente integrador, a ferramenta `buscar_manual_engenharia` já recebe uma consulta reescrita pelo próprio agente; a avaliação de ponta a ponta no LangSmith não mostrou ganho de corretude com a camada multi-query + HyDE e mediu ~7,6 s a mais por resposta, por isso ali o padrão passou a ser só a reescrita do agente (a camada continua disponível com `RAG_TRANSFORMACAO=multi-query+hyde`). Na configuração original, a ferramenta aplicava multi-query + HyDE por padrão. Custo: duas chamadas ao modelo auxiliar (Claude Haiku) por busca, ~2–3 s — aceitável frente ao custo de uma resposta errada sobre uma norma. A variável `RAG_TRANSFORMACAO=nenhuma` desliga a transformação, o que permite comparar as duas configurações como experimentos no LangSmith (Fase 5).

**Limites:** o gabarito tem 14 casos (amostra pequena); as transformações são estocásticas; e a causa do `CHK_012` (chunk com dois assuntos) só seria eliminada na origem, reindexando — registrado como melhoria futura para o "pipeline de outro bloco".